# Purview Migration Toolkit - Fabric Lakehouse Workflow

This notebook orchestrates Microsoft Purview artifact migration between tenants, storing all data and reports in the Lakehouse Files area.

**Prerequisites:**
- Access to source and target Purview accounts
- Azure CLI logged in or managed identity available
- Python 3.10+
- Fabric Lakehouse with Files area enabled

**Workflow:**
1. Export source Purview artifacts to manifest
2. Import into target Purview (dry-run + apply)
3. Generate relink plan
4. Apply relink plan (dry-run + apply)
5. Generate status reports (CSV/JSON)
6. Review results in Files area

## Section 1: Configure Lakehouse File Paths

Define the base path for the Lakehouse Files area and subdirectories for organizing manifests, plans, and reports.

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

# Lakehouse Files base path
LAKEHOUSE_FILES = "/lakehouse/default/Files"

# Create subdirectory structure
MANIFESTS_DIR = os.path.join(LAKEHOUSE_FILES, "purview_migration", "manifests")
PLANS_DIR = os.path.join(LAKEHOUSE_FILES, "purview_migration", "plans")
REPORTS_DIR = os.path.join(LAKEHOUSE_FILES, "purview_migration", "reports")
LOGS_DIR = os.path.join(LAKEHOUSE_FILES, "purview_migration", "logs")

# File paths for key artifacts
SOURCE_MANIFEST_PATH = os.path.join(MANIFESTS_DIR, "source-export.json")
TARGET_STATUS_PATH = os.path.join(MANIFESTS_DIR, "target-import-status.json")
RELINK_PLAN_PATH = os.path.join(PLANS_DIR, "relink-plan.json")
RELINK_STATUS_PATH = os.path.join(PLANS_DIR, "relink-status.json")
REPORT_JSON_PATH = os.path.join(REPORTS_DIR, "relink-report.json")
REPORT_CSV_PREFIX = os.path.join(REPORTS_DIR, "relink-report")

print("✓ Lakehouse file paths configured:")
print(f"  Manifests: {MANIFESTS_DIR}")
print(f"  Plans: {PLANS_DIR}")
print(f"  Reports: {REPORTS_DIR}")
print(f"  Logs: {LOGS_DIR}")

## Section 2: Initialize Lakehouse Connection

Verify the Lakehouse connection and Files area accessibility using NotebookUtils.

In [ ]:
try:
    import mssparkutils
    from mssparkutils.fs import *
    
    # Verify Lakehouse is accessible
    lakehouse_info = mssparkutils.notebook.run("", "")
    print("✓ Lakehouse connection verified")
    print(f"✓ Files area accessible at: {LAKEHOUSE_FILES}")
    
    # Get runtime context
    import notebookutils
    context = notebookutils.runtime.context
    print(f"✓ Notebook context: {context}")
    
except Exception as e:
    print(f"⚠ Warning: Lakehouse context not fully available: {e}")
    print("  This is normal outside Fabric environment. Using standard file I/O.")

## Section 3: Create Directory Structure in Files Area

Create the required folder hierarchy for organizing migration artifacts.

In [ ]:
def ensure_dirs():
    """Create directory structure using pathlib."""
    for dir_path in [MANIFESTS_DIR, PLANS_DIR, REPORTS_DIR, LOGS_DIR]:
        Path(dir_path).mkdir(parents=True, exist_ok=True)
        print(f"✓ Ensured directory: {dir_path}")

ensure_dirs()

# Try to list root directory to verify
try:
    files = mssparkutils.fs.ls(LAKEHOUSE_FILES)
    print(f"\n✓ Files area contents ({len(files)} items):")
    for f in files[:5]:
        print(f"  {f.name}")
    if len(files) > 5:
        print(f"  ... and {len(files) - 5} more")
except Exception as e:
    print(f"Note: Unable to list files with mssparkutils (OK outside Fabric): {type(e).__name__}")

## Section 4: Install and Initialize Purview Migration Toolkit

Install the Python migration toolkit from the repository.

In [ ]:
import subprocess

# Install purview-migration-toolkit from local repo or git
print("Installing Purview Migration Toolkit...")
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 
         "git+https://github.com/yourusername/purview-migration.git@main"],
        capture_output=True,
        text=True,
        timeout=120
    )
    if result.returncode != 0:
        print("Attempting local installation from current environment...")
        # If git not available, use editable install from relative path
        repo_root = Path.cwd().parent.parent  # Navigate to repo root
        if (repo_root / "src" / "purview_migration").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_root)], 
                          capture_output=True, timeout=120)
except Exception as e:
    print(f"⚠ Toolkit install: {e}")

# Import the migration modules
try:
    from purview_migration.exporter import export_manifest
    from purview_migration.importer import import_manifest
    from purview_migration.relink import build_relink_plan
    from purview_migration.relink_executor import apply_relink_plan
    from purview_migration.report_generator import export_report
    from purview_migration.io_utils import read_json, write_json
    
    print("✓ Purview Migration Toolkit imported successfully")
except ImportError as e:
    print(f"✗ Failed to import toolkit: {e}")
    print("Ensure toolkit is installed or available in Python path")

## Configuration: Set Your Purview Account Names

Edit these values before running the migration workflow.

In [ ]:
# ============================================================================
# CONFIGURATION: Edit these values for your migration
# ============================================================================

SOURCE_PURVIEW_ACCOUNT = "source-purview-account"  # e.g., "my-prod-purview"
TARGET_PURVIEW_ACCOUNT = "target-purview-account"  # e.g., "my-adlz-purview"

# Entity validation limit (increase for large estates)
MAX_ENTITY_VALIDATION = 2000

# Flags for workflow execution
RUN_IMPORT_DRY_RUN = True          # Run import in dry-run mode first
RUN_IMPORT_APPLY = False           # Set to True to apply imports
RUN_RELINK_DRY_RUN = True          # Run relink in dry-run mode first
RUN_RELINK_APPLY = False           # Set to True to apply relink
GENERATE_REPORTS = True            # Generate CSV/JSON reports
REPORT_FORMAT = "both"             # "json", "csv", or "both"

print("=" * 70)
print("PURVIEW MIGRATION CONFIGURATION")
print("=" * 70)
print(f"Source Purview Account:  {SOURCE_PURVIEW_ACCOUNT}")
print(f"Target Purview Account:  {TARGET_PURVIEW_ACCOUNT}")
print(f"Max Entity Validation:   {MAX_ENTITY_VALIDATION}")
print(f"Import Dry-Run:          {RUN_IMPORT_DRY_RUN}")
print(f"Import Apply:            {RUN_IMPORT_APPLY}")
print(f"Relink Dry-Run:          {RUN_RELINK_DRY_RUN}")
print(f"Relink Apply:            {RUN_RELINK_APPLY}")
print(f"Generate Reports:        {GENERATE_REPORTS} ({REPORT_FORMAT})")
print("=" * 70)

## Step 1: Export Source Purview Artifacts

Enumerate all tenant-level artifacts from the source Purview account and save to manifest.

In [ ]:
print("\n" + "="*70)
print("STEP 1: EXPORT SOURCE PURVIEW ARTIFACTS")
print("="*70)

try:
    print(f"\nExporting from: {SOURCE_PURVIEW_ACCOUNT}")
    manifest = export_manifest(SOURCE_PURVIEW_ACCOUNT, max_entities=2000)
    
    # Write manifest to Lakehouse
    write_json(SOURCE_MANIFEST_PATH, manifest.to_dict())
    print(f"✓ Manifest exported to: {SOURCE_MANIFEST_PATH}")
    
    # Display summary
    print("\n📊 Export Summary:")
    print(f"  Collections:       {len(manifest.collections)}")
    print(f"  Data Sources:      {len(manifest.data_sources)}")
    print(f"  Scans:             {sum(len(x) for x in manifest.scans_by_source.values())}")
    print(f"  Glossary Categories: {len(manifest.glossary_categories)}")
    print(f"  Glossary Terms:    {len(manifest.glossary_terms)}")
    print(f"  Classifications:   {len(manifest.classifications)}")
    print(f"  Scan Rulesets:     {len(manifest.scan_rulesets)}")
    print(f"  Scan Credentials:  {len(manifest.scan_credentials)}")
    print(f"  Entities:          {len(manifest.entities)}")
    
    if manifest.warnings:
        print(f"\n⚠ {len(manifest.warnings)} warnings during export:")
        for w in manifest.warnings[:5]:
            print(f"  • {w}")
        if len(manifest.warnings) > 5:
            print(f"  ... and {len(manifest.warnings) - 5} more")
    else:
        print("\n✓ No warnings during export")
        
except Exception as e:
    print(f"✗ Export failed: {e}")
    import traceback
    traceback.print_exc()

## Step 2: Import to Target Purview

First run a dry-run import to validate, then optionally apply changes.

In [ ]:
print("\n" + "="*70)
print("STEP 2: IMPORT TO TARGET PURVIEW")
print("="*70)

try:
    # Load source manifest
    manifest_data = read_json(SOURCE_MANIFEST_PATH)
    from purview_migration.models import MigrationManifest
    manifest = MigrationManifest.from_dict(manifest_data)
    
    # Dry-run import
    if RUN_IMPORT_DRY_RUN:
        print(f"\n📋 Running DRY-RUN import to: {TARGET_PURVIEW_ACCOUNT}")
        result_dryrun = import_manifest(TARGET_PURVIEW_ACCOUNT, manifest, dry_run=True)
        print(f"✓ Dry-run complete")
        print(f"  Skipped (would process): {result_dryrun.skipped}")
        print(f"  Warnings: {len(result_dryrun.warnings)}")
        if result_dryrun.warnings:
            for w in result_dryrun.warnings[:3]:
                print(f"    • {w}")
    
    # Apply import if requested
    if RUN_IMPORT_APPLY:
        print(f"\n📨 Applying import to: {TARGET_PURVIEW_ACCOUNT}")
        result_apply = import_manifest(TARGET_PURVIEW_ACCOUNT, manifest, dry_run=False)
        print(f"✓ Import completed")
        print(f"  Created: {result_apply.created}")
        print(f"  Updated: {result_apply.updated}")
        print(f"  Failed: {result_apply.failed}")
        
        # Save result status
        write_json(TARGET_STATUS_PATH, {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "target_account": TARGET_PURVIEW_ACCOUNT,
            "result": result_apply.as_dict()
        })
        print(f"✓ Status saved to: {TARGET_STATUS_PATH}")
    else:
        print("\n⏭ Import --apply not enabled. Skipping actual import.")
        print("  To apply: Set RUN_IMPORT_APPLY = True in Configuration cell")
        
except Exception as e:
    print(f"✗ Import failed: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Generate Relink Plan

Create a name-based mapping plan that links artifacts between source and target.

In [ ]:
print("\n" + "="*70)
print("STEP 3: GENERATE RELINK PLAN")
print("="*70)

try:
    # Load manifest
    manifest_data = read_json(SOURCE_MANIFEST_PATH)
    manifest = MigrationManifest.from_dict(manifest_data)
    
    # Generate relink plan
    plan = build_relink_plan(manifest)
    print(f"\n📋 Relink plan generated")
    
    # Save plan
    write_json(RELINK_PLAN_PATH, plan)
    print(f"✓ Plan saved to: {RELINK_PLAN_PATH}")
    
    # Display summary
    print(f"\n📊 Relink Plan Summary:")
    for artifact_type, count in plan["summary"].items():
        print(f"  {artifact_type:.<30} {count}")
    
    # Show notes
    if plan.get("notes"):
        print("\n📌 Plan Notes:")
        for note in plan["notes"]:
            print(f"  • {note}")
            
except Exception as e:
    print(f"✗ Relink plan generation failed: {e}")
    import traceback
    traceback.print_exc()

## Step 4: Apply Relink Plan

Validate and optionally apply the relink mappings to create/link artifacts in target.

In [ ]:
print("\n" + "="*70)
print("STEP 4: APPLY RELINK PLAN")
print("="*70)

try:
    # Load relink plan
    plan = read_json(RELINK_PLAN_PATH)
    
    # Dry-run relink
    if RUN_RELINK_DRY_RUN:
        print(f"\n📋 Running DRY-RUN relink against: {TARGET_PURVIEW_ACCOUNT}")
        result_dryrun = apply_relink_plan(
            TARGET_PURVIEW_ACCOUNT,
            plan,
            dry_run=True,
            max_entity_validation=MAX_ENTITY_VALIDATION
        )
        print(f"✓ Dry-run complete")
        print(f"  Linked:     {result_dryrun.linked}")
        print(f"  Unresolved: {result_dryrun.unresolved}")
        print(f"  Skipped:    {result_dryrun.skipped}")
        print(f"  Failed:     {result_dryrun.failed}")
        if result_dryrun.warnings:
            print(f"\n⚠ {len(result_dryrun.warnings)} warnings:")
            for w in result_dryrun.warnings[:3]:
                print(f"    • {w}")
    
    # Apply relink if requested
    if RUN_RELINK_APPLY:
        print(f"\n🔗 Applying relink to: {TARGET_PURVIEW_ACCOUNT}")
        result_apply = apply_relink_plan(
            TARGET_PURVIEW_ACCOUNT,
            plan,
            dry_run=False,
            max_entity_validation=MAX_ENTITY_VALIDATION
        )
        print(f"✓ Relink completed")
        print(f"  Created:    {result_apply.created}")
        print(f"  Linked:     {result_apply.linked}")
        print(f"  Unresolved: {result_apply.unresolved}")
        print(f"  Failed:     {result_apply.failed}")
        
        # Save updated plan status
        write_json(RELINK_STATUS_PATH, plan)
        print(f"✓ Updated plan saved to: {RELINK_STATUS_PATH}")
    else:
        print("\n⏭ Relink --apply not enabled. Skipping actual relink.")
        print("  To apply: Set RUN_RELINK_APPLY = True in Configuration cell")
        
except Exception as e:
    print(f"✗ Relink apply failed: {e}")
    import traceback
    traceback.print_exc()

## Step 5: Generate Status Reports

Export relink results grouped by outcome (JSON/CSV) for easy analysis.

In [ ]:
print("\n" + "="*70)
print("STEP 5: GENERATE STATUS REPORTS")
print("="*70)

try:
    if not GENERATE_REPORTS:
        print("\n⏭ Report generation disabled")
        print("  To enable: Set GENERATE_REPORTS = True in Configuration cell")
    else:
        # Load latest plan (with or without status updates)
        plan_path = RELINK_STATUS_PATH if Path(RELINK_STATUS_PATH).exists() else RELINK_PLAN_PATH
        plan = read_json(plan_path)
        
        # Generate JSON report
        if REPORT_FORMAT in ["json", "both"]:
            print(f"\n📄 Generating JSON report...")
            export_report(plan, REPORT_JSON_PATH, format_type="json")
            print(f"✓ JSON report: {REPORT_JSON_PATH}")
        
        # Generate CSV reports
        if REPORT_FORMAT in ["csv", "both"]:
            print(f"\n📊 Generating CSV reports...")
            export_report(plan, REPORT_CSV_PREFIX, format_type="csv")
            
            # List CSV files created
            csv_dir = Path(REPORT_CSV_PREFIX).parent
            csv_files = list(csv_dir.glob("relink-*.csv"))
            print(f"✓ Created {len(csv_files)} CSV files:")
            for csv_file in sorted(csv_files):
                size = csv_file.stat().st_size
                print(f"  • {csv_file.name} ({size:,} bytes)")
        
        print("\n✓ Reports generated and saved to Files area")
        print(f"  Location: {REPORTS_DIR}")
        
        # Display report summary
        if REPORT_FORMAT in ["json", "both"] and Path(REPORT_JSON_PATH).exists():
            with open(REPORT_JSON_PATH) as f:
                report = json.load(f)
            print("\n📋 Report Summary:")
            for status, count in report.get("summary", {}).get("byStatus", {}).items():
                print(f"  {status:.<20} {count}")
                
except Exception as e:
    print(f"✗ Report generation failed: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Review and Manage Files in Lakehouse

List, examine, and manage all migration artifacts stored in the Files area.

In [ ]:
print("\n" + "="*70)
print("STEP 6: LIST MIGRATION ARTIFACTS IN FILES AREA")
print("="*70)

def list_directory_contents(directory, max_depth=3, current_depth=0):
    """Recursively list directory contents with file sizes."""
    if current_depth >= max_depth:
        return
    
    try:
        dir_path = Path(directory)
        if not dir_path.exists():
            print(f"  (Directory doesn't exist yet: {directory})")
            return
        
        items = sorted(dir_path.iterdir())
        for item in items:
            indent = "  " * (current_depth + 1)
            if item.is_file():
                size = item.stat().st_size
                size_str = f"{size:,} B" if size < 1024 else f"{size/1024:.1f} KB"
                print(f"{indent}📄 {item.name:.<35} {size_str:>10}")
            elif item.is_dir():
                print(f"{indent}📁 {item.name}/")
                list_directory_contents(item, max_depth, current_depth + 1)
    except Exception as e:
        print(f"  Error listing directory: {e}")

# List migration artifacts
for category, dir_path in [
    ("Manifests", MANIFESTS_DIR),
    ("Plans", PLANS_DIR),
    ("Reports", REPORTS_DIR),
    ("Logs", LOGS_DIR)
]:
    print(f"\n{category}:")
    list_directory_contents(dir_path, max_depth=2)

# Display file statistics
print("\n" + "="*70)
print("FILE STATISTICS")
print("="*70)

try:
    # Count and summarize artifacts
    artifact_dirs = {
        "Manifests": MANIFESTS_DIR,
        "Plans": PLANS_DIR,
        "Reports": REPORTS_DIR
    }
    
    total_files = 0
    total_size = 0
    
    for category, dir_path in artifact_dirs.items():
        dir_obj = Path(dir_path)
        if dir_obj.exists():
            files = list(dir_obj.rglob("*"))
            file_count = len([f for f in files if f.is_file()])
            total_size_cat = sum(f.stat().st_size for f in files if f.is_file())
            
            total_files += file_count
            total_size += total_size_cat
            
            size_str = f"{total_size_cat / (1024*1024):.2f} MB" if total_size_cat > 1024*1024 else f"{total_size_cat / 1024:.1f} KB"
            print(f"{category:.<30} {file_count:>3} files, {size_str:>10}")
    
    total_size_str = f"{total_size / (1024*1024):.2f} MB" if total_size > 1024*1024 else f"{total_size / 1024:.1f} KB"
    print(f"{'Total':.<30} {total_files:>3} files, {total_size_str:>10}")
    
except Exception as e:
    print(f"Error getting statistics: {e}")

## Summary and Next Steps

✅ **Purview Migration Workflow Complete**

All artifacts have been exported, imported, relinked, and reported. Results are persisted in the Lakehouse Files area.

### Key Files Generated

- **Manifests**: Source artifact export with collection, data source, scan, glossary, and classification definitions
- **Plans**: Relink mapping plan showing source-to-target relationships
- **Reports**: Status summaries (JSON) and detailed groupings by outcome (CSV)

### Recommended Follow-Up Actions

1. **Review Unresolved Items**: Check reports for entities or collections that failed to relink
2. **Trigger Scans**: Re-run data source scans in target to update asset lineage
3. **Validate Domain Mappings**: Ensure governance domains are correctly mapped in target
4. **Verify Data Stewards**: Confirm role assignments are correctly transferred
5. **Archive Source**: After confirmation, archive source Purview to prevent duplicate work

## Utility: Clean Up Old Migration Files

Optional cell to remove old migration artifacts and start fresh.

In [ ]:
# ============================================================================
# CLEANUP: Uncomment and run to delete old migration files
# ============================================================================
# This is optional and useful if you're running multiple migrations
# and want to clean up previous artifacts.

CLEANUP_ENABLED = False  # Set to True to enable cleanup

if CLEANUP_ENABLED:
    print("Cleaning up old migration files...")
    
    try:
        # Optional: remove entire purview_migration directory and start fresh
        migration_root = Path(LAKEHOUSE_FILES) / "purview_migration"
        if migration_root.exists():
            import shutil
            shutil.rmtree(migration_root)
            print(f"✓ Deleted: {migration_root}")
            
        # Recreate directory structure
        ensure_dirs()
        print("✓ Directory structure recreated")
        
    except Exception as e:
        print(f"✗ Cleanup failed: {e}")
else:
    print("⏭ Cleanup disabled")
    print("  To enable cleanup: Set CLEANUP_ENABLED = True and re-run this cell")